# Gemma 4 × PrestaShop — résultats rejouables

Recalcule les tableaux du papier depuis `eval/results.csv` (une ligne par condition × essai × bug).
Les verdicts viennent d'oracles Playwright cachés exécutés sur une boutique Docker remise à zéro (`bench/eval.py`) ; ce notebook ne fait que les agréger.

In [ ]:
import pandas as pd, numpy as np
import os
path = next(p for p in ['eval/results.csv', '../eval/results.csv', '/kaggle/input/gemma4-legacy-replay/results.csv'] if os.path.exists(p))
df = pd.read_csv(path)
df.head()

## Taux de résolution par condition et par essai

In [ ]:
t = df.groupby(['condition', 'essai']).agg(resolus=('resolu', 'sum'), bon_fichier=('bon_fichier', 'sum'), n=('pr', 'count'))
t

In [ ]:
m = t.groupby('condition').agg(resolus_moy=('resolus', 'mean'), bon_fichier_moy=('bon_fichier', 'mean'), essais=('n', 'count'))
m['taux'] = (m.resolus_moy / 33).round(3)
m.sort_values('taux')

## Écart apparié par bug contre A (bootstrap, IC 95 %)

In [ ]:
def paired(c, ref='A', n=5000, seed=0):
    a = df[df.condition == ref].groupby('pr').resolu.mean()
    b = df[df.condition == c].groupby('pr').resolu.mean()
    d = (b - a).dropna().values
    rng = np.random.default_rng(seed)
    boots = np.sort([rng.choice(d, len(d)).mean() for _ in range(n)])
    return round(d.mean(), 3), round(boots[int(.025 * n)], 3), round(boots[int(.975 * n)], 3), len(d)

pd.DataFrame({c: paired(c) for c in df.condition.unique() if c != 'A'}, index=['écart', 'IC bas', 'IC haut', 'bugs']).T

## Bugs jamais / toujours résolus

In [ ]:
p = df.pivot_table(index='pr', columns='condition', values='resolu', aggfunc='mean')
p.sort_values('A')